# 00 - Load Data

Load and explore the Spanish split of NTRLab MediaSpeech from `data/NTRLabMediaSpeech/ES.tgz`.

In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
from pathlib import Path

import pandas as pd

from src import get_data

In [23]:
archive_path = Path("data/NTRLabMediaSpeech/ES.tgz")
dataset_dir = get_data.extract_mediaspeech_es(archive_path)
dataset_dir

WindowsPath('data/NTRLabMediaSpeech/extracted/ES')

In [24]:
index = get_data.load_mediaspeech_index(dataset_dir)
get_data.dataset_summary(index)

{'samples': 2507,
 'samples_30_60s': 0,
 'hours': 10.0,
 'min_seconds': 4.6,
 'median_seconds': 14.7,
 'max_seconds': 14.9,
 'with_transcription': 2507}

In [25]:
index.head()

,audio_id,audio_path,text_path,relative_path,duration_seconds,transcription
0,002f9994-f332-4240-a175-fdadce59d61f,data\NTRLabMediaSpeech\extracted\ES\002f9994-f...,data\NTRLabMediaSpeech\extracted\ES\002f9994-f...,002f9994-f332-4240-a175-fdadce59d61f.flac,14.9,y te han hecho así el sistema sanitario nacion...
1,00402961-0c7c-4603-9948-97379d1ebdc9,data\NTRLabMediaSpeech\extracted\ES\00402961-0...,data\NTRLabMediaSpeech\extracted\ES\00402961-0...,00402961-0c7c-4603-9948-97379d1ebdc9.flac,14.7,nada más lo que se vio en las redes tuviste un...
2,00c012c3-f579-49e0-b8f3-def2015940ef,data\NTRLabMediaSpeech\extracted\ES\00c012c3-f...,data\NTRLabMediaSpeech\extracted\ES\00c012c3-f...,00c012c3-f579-49e0-b8f3-def2015940ef.flac,7.3,total rechazo esa comparecencia dijo que no fu...
3,00e0fd2e-3e17-4c85-8b9a-ce6ed34db8c7,data\NTRLabMediaSpeech\extracted\ES\00e0fd2e-3...,data\NTRLabMediaSpeech\extracted\ES\00e0fd2e-3...,00e0fd2e-3e17-4c85-8b9a-ce6ed34db8c7.flac,14.5,granjeros podrán viajar a las islas baleares o...
4,00faacb0-422f-4971-85e0-71b2a3775729,data\NTRLabMediaSpeech\extracted\ES\00faacb0-4...,data\NTRLabMediaSpeech\extracted\ES\00faacb0-4...,00faacb0-422f-4971-85e0-71b2a3775729.flac,14.9,tema serio al decir feminista involucra necesa...


In [26]:
index["duration_seconds"].describe()

count    2507.000000
mean       14.364579
std         1.028818
min         4.600000
25%        14.400000
50%        14.700000
75%        14.800000
max        14.900000
Name: duration_seconds, dtype: float64

In [27]:
preferred = get_data.filter_by_duration(index, min_seconds=30, max_seconds=60)
len(preferred), preferred[["audio_id", "duration_seconds", "transcription"]].head(10)

(0,
 Empty DataFrame
 Columns: [audio_id, duration_seconds, transcription]
 Index: [])

In [ ]:
# MediaSpeech ES clips are short. Build a  `n` `min_seconds`-`max_seconds` second real-audio bundles.
bundles = get_data.select_sample_bundles(index, n=3, min_seconds=30, max_seconds=60, seed=42)
bundle_index = get_data.export_sample_bundles(bundles)
bundle_index[["bundle_id", "duration_seconds", "clip_count", "source_audio_ids"]]

,bundle_id,duration_seconds,clip_count,source_audio_ids
0,mediaspeech_es_01,44.55,3,"[d5bf33da-35ac-4b69-9140-ae4fe8e3aced, 0597577..."
1,mediaspeech_es_02,39.35,3,"[832c93d5-297b-49b8-a71e-8b468a783c61, a3879a1..."
2,mediaspeech_es_03,42.35,3,"[10afef01-dea3-45cb-a5a0-2c87aa62200b, 5eddffd..."


In [29]:
bundle_index

,audio_path,duration_seconds,clip_count,source_audio_ids,transcription,bundle_id
0,data\NTRLabMediaSpeech\samples\mediaspeech_es_...,44.55,3,"[d5bf33da-35ac-4b69-9140-ae4fe8e3aced, 0597577...",me llama muchísimo la atención entonces cómo s...,mediaspeech_es_01
1,data\NTRLabMediaSpeech\samples\mediaspeech_es_...,39.35,3,"[832c93d5-297b-49b8-a71e-8b468a783c61, a3879a1...",otra de nuestras fronteras para quienes como n...,mediaspeech_es_02
2,data\NTRLabMediaSpeech\samples\mediaspeech_es_...,42.35,3,"[10afef01-dea3-45cb-a5a0-2c87aa62200b, 5eddffd...",andalucía en un intercambio de felinos hace cu...,mediaspeech_es_03


In [30]:
# Listen to one 30-60 second bundle.
# The helper renders an embedded WAV HTML player, which is more reliable in VS Code notebooks.
get_data.display_audio_sample(bundle_index.iloc[2])

,audio_path,duration_seconds,clip_count,source_audio_ids,transcription,bundle_id
0,data\NTRLabMediaSpeech\samples\mediaspeech_es_...,42.35,3,"[10afef01-dea3-45cb-a5a0-2c87aa62200b, 5eddffd...",andalucía en un intercambio de felinos hace cu...,mediaspeech_es_03


andalucía en un intercambio de felinos hace cuatro años los cuidadores temen lo que pueda ocurrir si juntan a la familia ya que el cachorro competiría con su padre por la atención de su madre y eso puede terminar en traje
firmado por el eurodiputado rumano dachan choros hace dos semanas cuando conducía entre bucarest y bruselas varios países están trabajando en la reapertura de sus fronteras pero la comisión europea insiste
en madrid el ejército ha convertido el centro de convenciones y fema en el mayor hospital del país en tan solo diez días hasta han instalado vías de oxígeno bajo el suelo


In [31]:
# Listen to all selected bundles.
get_data.display_samples(bundle_index)

,audio_path,duration_seconds,clip_count,source_audio_ids,transcription,bundle_id
0,data\NTRLabMediaSpeech\samples\mediaspeech_es_...,44.55,3,"[d5bf33da-35ac-4b69-9140-ae4fe8e3aced, 0597577...",me llama muchísimo la atención entonces cómo s...,mediaspeech_es_01


me llama muchísimo la atención entonces cómo se entiende el traslado si lo traslado si lo transmite la gente porque hay algunos países de latinoamérica que tienen mucho flujo de contacto con
por lo general tenemos un nivel de ingresos más bajo comparado con otros grupos  eso hace que primero nos toque salir más de la casa para poder conseguir nuestra comida nuestras necesidades y
a la luz de esta experiencia y conociendo todas las todos los golpes de estado en américa latina todos los golpes de estado no solamente estos golpes vivido que vos también lo sentiste en carne propia verdad entonces el


,audio_path,duration_seconds,clip_count,source_audio_ids,transcription,bundle_id
0,data\NTRLabMediaSpeech\samples\mediaspeech_es_...,39.35,3,"[832c93d5-297b-49b8-a71e-8b468a783c61, a3879a1...",otra de nuestras fronteras para quienes como n...,mediaspeech_es_02


otra de nuestras fronteras para quienes como nosotros sus clientes son extranjeros en un sesenta por ciento es importante saber cuándo podrán viajar a italia la semana pasada la comisión europea propuso un levantamiento gradual de la
que si bueno pues si si vienes a londres pasa a saludarnos que estaremos aquí si yo cansadísima y y ya van bien las entradas según me dicen entonces
preparándome para esta entrevista pensaba en en en grandes personajes suyos que fueran personas de edad entonces decido quién puede ser el conselleiro puede ser de pronto don rigoberto puede ser urania


,audio_path,duration_seconds,clip_count,source_audio_ids,transcription,bundle_id
0,data\NTRLabMediaSpeech\samples\mediaspeech_es_...,42.35,3,"[10afef01-dea3-45cb-a5a0-2c87aa62200b, 5eddffd...",andalucía en un intercambio de felinos hace cu...,mediaspeech_es_03


andalucía en un intercambio de felinos hace cuatro años los cuidadores temen lo que pueda ocurrir si juntan a la familia ya que el cachorro competiría con su padre por la atención de su madre y eso puede terminar en traje
firmado por el eurodiputado rumano dachan choros hace dos semanas cuando conducía entre bucarest y bruselas varios países están trabajando en la reapertura de sus fronteras pero la comisión europea insiste
en madrid el ejército ha convertido el centro de convenciones y fema en el mayor hospital del país en tan solo diez días hasta han instalado vías de oxígeno bajo el suelo


,audio_path,duration_seconds,clip_count,source_audio_ids,transcription,bundle_id
0,data\NTRLabMediaSpeech\samples\mediaspeech_es_...,44.55,3,"[d5bf33da-35ac-4b69-9140-ae4fe8e3aced, 0597577...",me llama muchísimo la atención entonces cómo s...,mediaspeech_es_01
1,data\NTRLabMediaSpeech\samples\mediaspeech_es_...,39.35,3,"[832c93d5-297b-49b8-a71e-8b468a783c61, a3879a1...",otra de nuestras fronteras para quienes como n...,mediaspeech_es_02
2,data\NTRLabMediaSpeech\samples\mediaspeech_es_...,42.35,3,"[10afef01-dea3-45cb-a5a0-2c87aa62200b, 5eddffd...",andalucía en un intercambio de felinos hace cu...,mediaspeech_es_03


In [32]:
# Persist selected bundles as reusable project samples under src/samples.
persisted_samples = get_data.persist_sample_bundles(bundles)
persisted_samples[["bundle_id", "audio_path", "duration_seconds", "clip_count", "license"]]

,bundle_id,audio_path,duration_seconds,clip_count,license
0,mediaspeech_es_01,src\samples\mediaspeech_es_01.flac,44.55,3,CC BY 4.0
1,mediaspeech_es_02,src\samples\mediaspeech_es_02.flac,39.35,3,CC BY 4.0
2,mediaspeech_es_03,src\samples\mediaspeech_es_03.flac,42.35,3,CC BY 4.0


In [33]:
# Read persisted samples later without rebuilding the dataset index.
loaded_samples = get_data.load_persisted_samples()
loaded_samples[["bundle_id", "audio_path", "duration_seconds", "clip_count", "license"]]

,bundle_id,audio_path,duration_seconds,clip_count,license
0,mediaspeech_es_01,src\samples\mediaspeech_es_01.flac,44.55,3,CC BY 4.0
1,mediaspeech_es_02,src\samples\mediaspeech_es_02.flac,39.35,3,CC BY 4.0
2,mediaspeech_es_03,src\samples\mediaspeech_es_03.flac,42.35,3,CC BY 4.0


In [34]:
# Dataset citation/license details to keep with reports or demos.
get_data.display_mediaspeech_license_info()

### Dataset attribution
- Dataset: NTRLab MediaSpeech, Spanish split (ES)
- Source: https://www.openslr.org/108/
- Official ES download: https://www.openslr.org/resources/108/ES.tgz
- License: CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/)
- Attribution: Use with attribution to the MediaSpeech dataset/NTR Labs and include the OpenSLR source URL. CC BY 4.0 permits sharing and adaptation, including for commercial use, when appropriate credit is provided.
- Notes: OpenSLR describes MediaSpeech as short speech segments automatically extracted from media videos available on YouTube and manually transcribed. This project uses locally derived bundle samples created by concatenating real clips from the Spanish split; keep source_audio_ids in outputs so samples remain traceable to the original files.

```bibtex
@misc{mediaspeech2021,
      title={MediaSpeech: Multilanguage ASR Benchmark and Dataset},
      author={Rostislav Kolobov and Olga Okhapkina and Olga Omelchishina, Andrey Platunov and Roman Bedyakin and Vyacheslav Moshkin and Dmitry Menshikov and Nikolay Mikhaylovskiy},
      year={2021},
      eprint={2103.16193},
      archivePrefix={arXiv},
      primaryClass={eess.AS}
}
```

{'dataset': 'NTRLab MediaSpeech, Spanish split (ES)',
 'source': 'https://www.openslr.org/108/',
 'license': 'CC BY 4.0',
 'license_url': 'https://creativecommons.org/licenses/by/4.0/',
 'official_download': 'https://www.openslr.org/resources/108/ES.tgz',
 'attribution': 'Use with attribution to the MediaSpeech dataset/NTR Labs and include the OpenSLR source URL. CC BY 4.0 permits sharing and adaptation, including for commercial use, when appropriate credit is provided.',
 'notes': 'OpenSLR describes MediaSpeech as short speech segments automatically extracted from media videos available on YouTube and manually transcribed. This project uses locally derived bundle samples created by concatenating real clips from the Spanish split; keep source_audio_ids in outputs so samples remain traceable to the original files.',
 'citation': '@misc{mediaspeech2021,\n      title={MediaSpeech: Multilanguage ASR Benchmark and Dataset},\n      author={Rostislav Kolobov and Olga Okhapkina and Olga Omelch